
# Smart Movie Recommendation System

Most basic recommendation systems crash if you type a movie title incorrectly or look for something missing from their database.

This **Smart Movie Recommendation System** is built to think like a human assistant. If it can't find your exact movie request, it automatically checks for spelling typos to suggest what you meant. If that still fails, it falls back to recommending movies within your preferred genre so you do not see an error screen.

## Setting Up and Loading the Data
Before we can build anything, we need to bring in our specialized tools (like Pandas for handling data tables and Scikit-Learn for the math). We then load our movie file and take a quick look at its structure to make sure all our columns, like titles and genres, are present and correct.

In [1]:
# import the neccessary library

import pandas as pd
import numpy as np
import difflib

In [2]:
# import and the dataset

movies = pd.read_csv("movies.csv")
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [3]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10329 entries, 0 to 10328
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  10329 non-null  int64 
 1   title    10329 non-null  object
 2   genres   10329 non-null  object
dtypes: int64(1), object(2)
memory usage: 242.2+ KB


## Cleaning Up the Movie Genres

Raw data is often messy. In this step, cleaned up the `genres` column by removing the unnecessary symbols `|`. This leaves us with clean, readable text tags (like "Action Comedy") that computers can easily compare later on.

In [4]:
movies["genres"] = movies["genres"].str.replace("|", " ", regex=False).str.lower()
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),adventure animation children comedy fantasy
1,2,Jumanji (1995),adventure children fantasy
2,3,Grumpier Old Men (1995),comedy romance
3,4,Waiting to Exhale (1995),comedy drama romance
4,5,Father of the Bride Part II (1995),comedy


## Turning Genres into Mathematical Values

Computers cannot read or understand words like "Action" or "Sci-Fi" the way humans do; they only understand numbers.

Here, we use a tool called a **Vectorizer** to count how often each genre appears across our entire dataset. It ignores common, unhelpful filler words (like "the" or "and") and transforms our text genres into a massive grid of numerical weights. The final `print` command shows us the total size of this newly created number grid.

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(movies["genres"])

print(tfidf_matrix.shape)

(10329, 23)


## Calculating Movie Connections (The Similarity Map)
Now that our movie genres are represented as numbers, we can calculate how similar the movies are to one another.

Using a metric called **Cosine Similarity**, the computer compares the numerical fingerprint of every single movie against every other movie in the dataset. If two movies share almost identical genres, they get a score close to 1; if they have nothing in common, they get a 0. This creates a giant connection map that our system will use to fetch recommendations instantly.

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)


## Building the "Smart" Search Engine
This is the core brain of our application. Instead of just doing a basic search, this function is designed to handle user mistakes and prevent crashes by following a smart 4-step decision tree:

1. **Perfect Match:** If the user types the title correctly, it looks up our connection map and immediately grabs the top 10 most similar movies.
2. **"Did You Mean?":** If there is a typo (like "Dark Knight Rise"), it automatically finds the closest matching title in the dataset and uses that instead.
3. **Genre Backup:** If the title completely fails but an optional genre was provided (like "Sci-Fi"), it pulls the top matching movies from that category.
4. **Deep Fallback:** If the user accidentally typed a genre name directly into the movie title box, the system realizes the mistake and shows movies for that genre so it never shows an error screen.

In [7]:

def get_smart_recommendation(title, genre=None, movies_df=movies, cosine_sim=cosine_sim):
    title_clean = title.strip()
    all_titles = movies_df['title'].tolist()

    # Look for an exact match
    if title_clean in all_titles:
        target_title = title_clean
    else:
        # Suggestion Logic: Find the closest match in the dataset
        matches = difflib.get_close_matches(title_clean, all_titles, n=1, cutoff=0.6)

        if matches:
            target_title = matches[0]
            print(f"'{title_clean}' not found. Did you mean '{target_title}'?")
        else:
            # Genre fallback: If no close title match, try the genre
            if genre:
                genre_matches = movies_df[movies_df['genres'].str.contains(genre, case=False, na=False)]
                if not genre_matches.empty:
                    print(f"No title match for '{title_clean}'. Showing '{genre}' movies:")
                    return genre_matches['title'].head(10).reset_index(drop=True)
            return "Movie not found. Try checking your spelling or entering a genre!"

    # 4. Run the recommendation logic on the 'target_title' (either the exact or the suggested one)
    idx = movies_df[movies_df['title'] == target_title].index[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    movie_indices = [i[0] for i in sim_scores[1:11]]

    return movies_df['title'].iloc[movie_indices].reset_index(drop=True)

## Testing the `get_smart_recommendation` function

In [8]:
print(get_smart_recommendation("Toy Story (1995)"))

0                                          Antz (1998)
1                                   Toy Story 2 (1999)
2       Adventures of Rocky and Bullwinkle, The (2000)
3                     Emperor's New Groove, The (2000)
4                                Monsters, Inc. (2001)
5    DuckTales: The Movie - Treasure of the Lost La...
6                                     Wild, The (2006)
7                               Shrek the Third (2007)
8                       Tale of Despereaux, The (2008)
9    Asterix and the Vikings (Astérix et les Viking...
Name: title, dtype: object


In [9]:
print(get_smart_recommendation("Merlin"))

Movie not found. Try checking your spelling or entering a genre!


In [10]:
print(get_smart_recommendation("Merlin", "Action"))

No title match for 'Merlin'. Showing 'Action' movies:
0                                  Heat (1995)
1                          Sudden Death (1995)
2                             GoldenEye (1995)
3                      Cutthroat Island (1995)
4                           Money Train (1995)
5                             Assassins (1995)
6                       Dead Presidents (1995)
7                         Mortal Kombat (1995)
8    Lawnmower Man 2: Beyond Cyberspace (1996)
9                   From Dusk Till Dawn (1996)
Name: title, dtype: object


In [11]:
print(get_smart_recommendation("oy Story"))

'oy Story' not found. Did you mean 'Toy Story (1995)'?
0                                          Antz (1998)
1                                   Toy Story 2 (1999)
2       Adventures of Rocky and Bullwinkle, The (2000)
3                     Emperor's New Groove, The (2000)
4                                Monsters, Inc. (2001)
5    DuckTales: The Movie - Treasure of the Lost La...
6                                     Wild, The (2006)
7                               Shrek the Third (2007)
8                       Tale of Despereaux, The (2008)
9    Asterix and the Vikings (Astérix et les Viking...
Name: title, dtype: object


## Freezing and Saving the recommendation system
Calculating the similarity map between thousands of movies takes a lot of time and computer horsepower. We don't want our final website to recalculate all of these mathematical connections every single time a user opens the page.

To solve this, we use a tool called **Pickle** to "freeze" our finished dataset and similarity map into a single file (`smart_movies_recommendation.pkl`). Our web application can read this saved file instantly, making the user experience incredibly fast.

In [12]:
import pickle

with open(" smart_movies_recommendation.pkl", "wb") as file:
  pickle.dump((movies, cosine_sim), file)